## Explanations for observation with the worst prediction

This notebook provides some explanations of final model for observation from test dataset for which difference between real value and predicted value was the biggest.

### Import packages

In [ ]:
import functions as fun
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import dalex as dx
import lime
import lime.lime_tabular

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

### Load model to be explain

In [ ]:
pickle_in = open(".\..\models\pickle\MEPS_xgb_model_final_v2.pickle", "rb")
reg_xgb = pickle.load(pickle_in)

### Read data

In [ ]:
path = ".\..\data\MEPS_data_preprocessed"
X_train, y_train = fun.read_x_y(path + "_train.csv", "HEALTHEXP")
X_test, y_test = fun.read_x_y(path + "_test.csv", "HEALTHEXP")

In [ ]:
# All data (train + test)
X = X_train.copy()
X = X.append(X_test, ignore_index = True)

y = y_train.copy()
y = y.append(y_test, ignore_index = True)

In [ ]:
raw_test_data = pd.read_csv(path + "_test.csv")
X_test_raw = raw_test_data.drop("HEALTHEXP", axis = 1)
y_test_raw = raw_test_data["HEALTHEXP"]

### Find and explore observation with the worst prediction

In [ ]:
y_pred_test = reg_xgb.predict(X_test)
obs_idx = fun.find_nth_obs_idx(y_test, y_pred_test, -1)

In [ ]:
# Real value (target transformed with log base 3)
y_test[obs_idx]

In [ ]:
# Real value (raw)
y_test_raw[obs_idx]

In [ ]:
# Predicted value
y_pred_test[obs_idx]

In [ ]:
# Observation
obs = X_test.iloc[[obs_idx], :]
obs

**Meaning of variables and their values:** <br/>
* REGION = 1.0 	 - 	 census region was northeast
* AGE31X = 76.0 	 - 	 76 years old
* GENDER = 1.0 	 - 	 female
* RACE3 = 0.0 	 - 	 
* MARRY31X = 2.0 	 - 	 widowed
* EDRECODE = 13.0 	 - 	 GED or high school degree
* FTSTU31X = -1.0 	 - 	 student status - inapplicable
* ACTDTY31 = 4.0 	 - 	 military full-time active duty - over 59 - inapplicable
* HONRDC31 = 2.0 	 - 	 not honorably discharged from military
* RTHLTH31 = 5.0 	 - 	 perceived health status - poor
* MNHLTH31 = 4.0 	 - 	 perceived mental health status - fair
* HIBPDX = 1.0 	 - 	 high blood pressure diagnosed
* CHDDX = 1.0 	 - 	 coronary heart disease diagnosed
* ANGIDX = 2.0 	 - 	 angina wasn't diagnosed
* MIDX = 2.0 	 - 	 heart attack wasn't diagnosed
* OHRTDX = 2.0 	 - 	 any other heart diseases weren't diagnosed
* STRKDX = 2.0 	 - 	 stroke wasn't diagnosed
* EMPHDX = 1.0 	 - 	 emphysema diagnosed
* CHBRON31 = 1.0 	 - 	 chronic bronchitis diagnosed
* CHOLDX = 1.0 	 - 	 high cholesterol diagnosed
* CANCERDX = 1.0 	 - 	 cancer diagnosed
* DIABDX = 2.0 	 - 	 diabetes wasn't diagnosed
* JTPAIN31 = 1.0 	 - 	 joint pain last 12 months diagnosed
* ARTHDX = 1.0 	 - 	 arthritis diagnosed
* ARTHTYPE = 1.0 	 - 	 type of arthritis - rheumatoid
* ASTHDX = 2.0 	 - 	 asthma wasn't diagnosed
* ADHDADDX = -1.0 	 - 	 ADHD or ADD diagnosis - inapplicable
* PREGNT31 = -1.0 	 - 	 pregnant - inapplicable
* WLKLIM31 = 1.0 	 - 	 has limitation in physical functioning
* ACTLIM31 = 1.0 	 - 	 has any other limitation work/house work/school
* SOCLIM31 = 1.0 	 - 	 has social limitation
* COGLIM31 = 1.0 	 - 	 has cognitive limitation
* DFHEAR42 = 2.0 	 - 	 hasn't serious difficulty hearing
* DFSEE42 = 1.0 	 - 	 has serious difficulty see or wears glasses
* ADSMOK42 = -1.0 	 - 	 doesn't smoke
* PCS42 = -1.0 	 - 	 saq:phy component summry sf-12v2 imputed - inapplicable
* MCS42 = -1.0 	 - 	 mnt component summry sf-12v2 imputed - inapplicable
* K6SUM42 = -1.0 	 - 	 overall rating of feelings - inapplicable (last 30 days)
* PHQ242 = -1.0 	 - 	 overall rating of feelings - inapplicable (last 2 weeks)
* EMPST31 = 4.0 	 - 	 employment status - ?
* POVCAT15 = 5.0 	 - 	 family income as % of poverty line - high income
* INSCOV15 = 2.0 	 - 	 health insurance coverage indicator 2015 - public only
* INCOME_M = 12345.0 - 	 person total income = 12345.0 

In [ ]:
diagnosis_names = [name for name in X_train.columns.values if name[-2:] == "DX"]
fn = lambda row: sum([row[n] for n in diagnosis_names if row[n] == 1])

print("Number of positive diagnoses: ", int(obs.apply(fn, axis=1)[obs_idx]))
print("Number of all diseases: ", len(diagnosis_names))

### Find and explore similar observarions

In [ ]:
# DataFrame with y_train and y_pred_train
y_pred_train = reg_xgb.predict(X_train)
y_train_pred_train_df = pd.DataFrame()
y_train_pred_train_df["y_train"] = y_train.values
y_train_pred_train_df["y_pred_train"] = y_pred_train

# Observations with target value equals to zero and predicted value greater than zero (train)
obs_with_pred_gt_zero = y_train_pred_train_df[(y_train_pred_train_df["y_train"] == 0) & (y_train_pred_train_df["y_pred_train"] > 0)].index.values
obs_with_pred_gt_zero_df = X_train.iloc[obs_with_pred_gt_zero, :].copy()

In [ ]:
# DataFrame with y and y_pred
y_pred = reg_xgb.predict(X)
y_df = pd.DataFrame()
y_df["y"] = y.values
y_df["y_pred"] = y_pred

# Observations with target value equals to zero and predicted value greater than zero (all data)
obs_with_pred_gt_zero_all = y_df[(y_df["y"] == 0) & (y_df["y_pred"] > 0)].index.values
obs_with_pred_gt_zero_all_df = X.iloc[obs_with_pred_gt_zero_all, :].copy()

In [ ]:
# Similar observations, i.e. people at the age of 74 - 78 who have ever been diagnosed with high blood pressure and cancer
indexes = X_train.query("AGE31X >= 74 and AGE31X <= 78 and CANCERDX == 1 and HIBPDX == 1").index.values
values = y_train[indexes]

In [ ]:
plt.figure(figsize = (8, 6))
plt.hist(values, bins = 20, rwidth = 0.9)
plt.grid(alpha = 0.5)
plt.title("Distribution of target values for observations in age of 74 - 78\nwith diagnosed cancer and high blood pressure", size = 15)
plt.xlabel("Target values", size = 14)
plt.ylabel("Count", size = 14)
plt.xticks(size = 13)
# plt.yticks(list(range(11)), size = 13)
plt.yticks(list(range(0, 23, 2)), size = 13)
plt.savefig("images/06-02-explanations_obs_hist.png")
plt.show()

In [ ]:
plt.figure(figsize = (8, 5))
plt.boxplot(values, vert = False)
plt.grid(alpha = 0.5)
plt.title("Distribution of target values for observations in age of 74 - 78\nwith diagnosed cancer and high blood pressure", size = 15)
plt.xlabel("Target values", size = 14)
plt.xticks(size = 13)
plt.yticks([])
plt.savefig("images/06-02-explanations_obs_boxplot.png")
plt.show()

### Explanations

#### 1) Break Down / Shap

In [ ]:
exp = dx.Explainer(reg_xgb, X_train, y_train, label = "MEPS XGB Pipeline")

In [ ]:
obs_bd = exp.predict_parts(obs, type = "break_down")
obs_bd_interactions = exp.predict_parts(obs, type = "break_down_interactions")

In [ ]:
obs_bd.plot(max_vars = 20)

In [ ]:
obs_bd_interactions.plot(max_vars = 20)

In [ ]:
obs_shap = exp.predict_parts(obs, type = "shap", B = 20)
obs_shap.plot(max_vars = 20)

In [ ]:
obs_shap2 = exp.predict_parts(obs, type = "shap")
obs_shap2.plot(max_vars = 20)

In [ ]:
obs_shap = exp.predict_parts(obs, type = "shap", B = 20)
obs_shap.plot(max_vars = 20)

In [ ]:
obs_shap3 = exp.predict_parts(obs, type = "shap", B = 100)
obs_shap3.plot(max_vars = 20)

In [ ]:
obs_shap4 = exp.predict_parts(obs, type = "shap", B = 200)
obs_shap4.plot(max_vars = 20)

#### 2) LIME

In [ ]:
# exp_lime = lime.lime_tabular.LimeTabularExplainer(X_train, 
#                                                     mode = "regression", 
#                                                     feature_names = X_train.columns.values,
#                                                     discretize_continuous = False,
#                                                     verbose = True, 
#                                                     random_state = 123)

In [ ]:
# obs_lime = exp_lime.explain_instance(obs, reg_xgb.predict)

In [ ]:
# obs_lime.show_in_notebook()

In [ ]:
feature_names = X_train.columns

In [ ]:
def explain_with_lime(model, index, num_f):
    explainer = lime.lime_tabular.LimeTabularExplainer(X_train.values, 
                                                       feature_names = feature_names,
                                                       class_names = ["HEALTHEXP"],
                                                       verbose = True, 
                                                       mode = "regression")
    exp = explainer.explain_instance(X_test.values[index], lambda x: model.predict(pd.DataFrame(x, columns = X_test.columns)), num_features = num_f)
    exp.show_in_notebook(show_table = True)
    file = f"06-{index}-explanations_lime.html"
    exp.save_to_file(file)

In [ ]:
explain_with_lime(reg_xgb, obs_idx, 5)

In [ ]:
# Features distributions of features that seems to be the most important according to LIME
fig = plt.figure(figsize = (16, 8))
for idx, colname in enumerate(["RTHLTH31", "PCS42", "POVCAT15", "AGE31X", "RACE3"]):
    i = idx + 1
    ax = fig.add_subplot(2, 3, i)
    x = obs_with_pred_gt_zero_df[colname].values
#     ax.boxplot(x, vert = False)
    ax.hist(x, bins = 10, rwidth = 0.9)
    ax.grid(alpha = 0.5)
    ax.set_title(colname, size = 14)
    for tick in ax.xaxis.get_major_ticks():
        tick.label.set_fontsize(12) 
    for tick in ax.yaxis.get_major_ticks():
        tick.label.set_fontsize(12) 
fig.savefig("images/06-features-distributions-for-features-from-lime.png")

In [ ]:
indexes2 = X.query("RTHLTH31 > 3 and POVCAT15 == 5 and AGE31X > 57 and RACE3 == 0").index.values
values2 = y[indexes2]

In [ ]:
plt.figure(figsize = (8, 6))
plt.hist(values2, bins = 20, rwidth = 0.9)
plt.grid(alpha = 0.5)
plt.title("Distribution of target values for observations at age > 57\nwith perceived health status fair or poor, high family income, white race", size = 15)
plt.xlabel("Target values", size = 14)
plt.ylabel("Count", size = 14)
plt.xticks(size = 13)
plt.yticks(size = 13)
plt.savefig("images/06-02-explanations_obs_hist2.png")
plt.show()

#### 3) Ceteris Paribus

In [ ]:
obs_cp = exp.predict_profile(obs)

In [ ]:
obs_cp.plot()

In [ ]:
obs_cp.plot(variables = ["AGE31X"])

In [ ]:
obs_cp.plot(variables = ["CANCERDX"])

In [ ]:
obs_cp.plot(variables = ["HIBPDX"])

#### 4) Permutational variable importance

In [ ]:
pvi = permutation_importance(reg_xgb, X_train, y_train, n_repeats = 10, random_state = 123)
fun.plot_pvi_boxplots(pvi, 10, X_train, "XGB model", "images/06-02-explanations_pvi")